# Capdyssey 품질 RF + AF 통합 연구용 v2

MIMIC+6종 AF 모델과 원본 손상 특징 RF를 연결합니다. 새 학습이 아니라 저장 모델 패키징·통합 추론 재현입니다.

**시작 전: 런타임 → 런타임 유형 변경 → T4 GPU.**

기존 시험군을 재사용한 개발 평가: MIMIC 전체 AF 검출률 96.62%, 판정 구간 특이도 79.84%, 오탐 51개. 6종 전체 AF 검출률 90.63%, 판정 구간 특이도 80.74%, 오탐 1,163개. 독립 검증·임상 검증·센서 일반화 완료 모델이 아닙니다.


In [ ]:
from pathlib import Path
import sys, subprocess, importlib.util
import torch
assert torch.cuda.is_available(), 'Colab T4 GPU로 연결하세요. AF 추론을 CPU로 대체하지 않습니다.'
packages={'wfdb':'wfdb','h5py':'h5py','sklearn':'scikit-learn','joblib':'joblib'}
missing=[package for module,package in packages.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
repo=Path('/content/capdyssey_quality_v2')
if not repo.exists(): subprocess.run(['git','clone','https://github.com/dung-ho03/capdyssey.git',str(repo)],check=True)
assert (repo/'scripts/export_quality_af_v2.py').exists(), '최신 통합 실행 코드가 필요합니다.'
print('실행 코드 커밋:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))
sys.path.insert(0,str(repo/'scripts'))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/Capdyssey/af_integrated')
assert (BASE/'audits/rf_raw_artifact_v1/0ea3e8d56aa7/frozen_candidate_development_test_v1/complete.json').exists(), '이 실험의 저장된 개발 평가 결과가 필요합니다.'


## 저장 모델 패키징 및 통합 코드 재현 확인

원본 데이터 재다운로드나 재학습은 하지 않습니다. 저장된 품질 점수와 AF 점수를 일부 기존 구간에서 재현하고 ZIP을 생성합니다. 성공 문구가 나오기 전에는 패키지 재현 확인이 완료된 것이 아닙니다.


In [ ]:
from export_quality_af_v2 import run
RELEASE=run(BASE)


## 팀 센서 연결용 입력 함수

정확히 10초 길이의 단일 **원본 PPG 채널**과 실제 샘플링 주파수를 전달합니다. ECG나 AF 정답을 입력으로 사용하지 않습니다. 이 함수가 동작해도 센서 데이터의 정확도가 검증된 것은 아닙니다.


In [ ]:
from quality_af_v2 import QualityAF
predictor=QualityAF(RELEASE)

def predict_ppg(raw_ppg, sampling_hz):
    return predictor.predict(raw_ppg, sampling_hz)

# 사용 예시(실제 파형을 준비한 뒤 실행):
# result = predict_ppg(raw_ppg=my_ppg_array, sampling_hz=100)
# print(result)
# status='remeasure'는 비AF가 아닙니다. label과 af_score는 None입니다.


In [ ]:
import pandas as pd
display(pd.read_csv(RELEASE/'development_test.csv'))
print('ZIP 위치:',RELEASE.with_suffix('.zip'))
# 다운로드가 필요할 때 주석을 해제하세요.
# from google.colab import files
# files.download(str(RELEASE.with_suffix('.zip')))
